# Imports

In [1]:
import os
os.environ['KERAS_BACKEND'] = 'tensorflow'

import keras
import numpy as np
import pandas as pd
import transformers
import tensorflow as tf
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import sentencepiece as spm

I0000 00:00:1795738005.624072  117735 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


# Choose the dataset

Get the dataset from [here](https://tatoeba.org/en/downloads). Preferably use russian to english translations.

In [2]:
dataset = pd.read_csv('dataset.tsv', sep='\t', header=None, usecols=[1, 3])
len(dataset)


830696

In [3]:
dataset.head()

,1,3
0,Один раз в жизни я делаю хорошее дело... И оно...,For once in my life I'm doing a good deed... A...
1,Давайте что-нибудь попробуем!,Let's try something.
2,Мне пора идти спать.,I have to go to sleep.
3,Что ты делаешь?,What are you doing?
4,Что ты делаешь?,What do you make?


In [4]:
dataset[-10:]

,1,3
830686,"Поверить не могу, что уже Рождество.",I can't believe it's Christmas already.
830687,Я ничего не готовила.,I wasn't cooking anything.
830688,Я ничего не готовил.,I wasn't cooking anything.
830689,Я сегодня не буду готовить.,Today I will not cook.
830690,Мне нужна сумка. Можете одолжить?,I need a bag. Will you lend me one?
830691,Мне нужна сумка. Можешь одолжить?,I need a bag. Will you lend me one?
830692,Мне нравятся шахматы.,I like chess.
830693,Она пожаловалась ему на еду.,She complained to him about the food.
830694,Все персонажи легко узнаваемы.,All the characters are easily recognizable.
830695,"«Кто эта женщина в белом платье?» — «Это Мэри,...","Who is the lady in the white dress? ""It's Mary..."


# Tokenizer

Train your own tokenizer on a corpus for the languages you've chosen or load a pretrained tokenizer. The tokenizer must allow BOS, EOS and PAD tokens.

In [5]:
# with open("tokenizer_corpus.txt", "w", encoding="utf-8") as f:
#     for el in tqdm(dataset.iterrows()):
#         f.write(el[1][1].strip() + '\n')
#         f.write(el[1][3].strip() + '\n')
        

In [3]:
tokenizers_folder = 'tokenizers/'

# spm.SentencePieceTrainer.train(
#     input='tokenizer_corpus.txt',
#     model_prefix=tokenizers_folder + 'ru_en_tokenizer8k',

#     model_type='unigram',
#     vocab_size=8000,

#     character_coverage=1.0,
#     byte_fallback=True,

#     pad_id=0,
#     unk_id=1,
#     bos_id=2,
#     eos_id=3,
# )

In [4]:
# tokenizer8k = spm.SentencePieceProcessor(
#     model_file=tokenizers_folder + 'ru_en_tokenizer8k.model'
# )
tokenizer16k = spm.SentencePieceProcessor(
    model_file=tokenizers_folder + 'ru_en_tokenizer16k.model'
)
# tokenizer32k = spm.SentencePieceProcessor(
#     model_file=tokenizers_folder + 'ru_en_tokenizer32k.model'
# )
# tokenizer48k = spm.SentencePieceProcessor(
#     model_file=tokenizers_folder + 'ru_en_tokenizer48k.model'
# )

text = 'Привет, how are you?'

tokens = tokenizer16k.encode(text, out_type=str)
ids = tokenizer16k.encode(text, out_type=int)

print(tokens)
print(ids)

print(tokenizer16k.decode(ids))

['▁Привет', ',', '▁how', '▁are', '▁you', '?']
[5563, 261, 493, 316, 266, 263]
Привет, how are you?


In [8]:
ru_tok_8k = [np.array(tokenizer8k.encode(i, out_type=int)) for i in tqdm(dataset[1])]
en_tok_8k = [np.array(tokenizer8k.encode(i, out_type=int)) for i in tqdm(dataset[3])]

ru_tok_16k = [np.array(tokenizer16k.encode(i, out_type=int)) for i in tqdm(dataset[1])]
en_tok_16k = [np.array(tokenizer16k.encode(i, out_type=int)) for i in tqdm(dataset[3])]

ru_tok_32k = [np.array(tokenizer32k.encode(i, out_type=int)) for i in tqdm(dataset[1])]
en_tok_32k = [np.array(tokenizer32k.encode(i, out_type=int)) for i in tqdm(dataset[3])]

ru_tok_48k = [np.array(tokenizer48k.encode(i, out_type=int)) for i in tqdm(dataset[1])]
en_tok_48k = [np.array(tokenizer48k.encode(i, out_type=int)) for i in tqdm(dataset[3])]

100%|██████████| 830696/830696 [00:04<00:00, 176839.46it/s]


In [9]:
ru_symbol_len = np.array(dataset[1].str.len())
en_symbol_len = np.array(dataset[3].str.len())

ru_tok_8k_len = np.array([len(t) for t in ru_tok_8k])
en_tok_8k_len = np.array([len(t) for t in en_tok_8k])

ru_tok_16k_len = np.array([len(t) for t in ru_tok_16k])
en_tok_16k_len = np.array([len(t) for t in en_tok_16k])

ru_tok_32k_len = np.array([len(t) for t in ru_tok_32k])
en_tok_32k_len = np.array([len(t) for t in en_tok_32k])

ru_tok_48k_len = np.array([len(t) for t in ru_tok_48k])
en_tok_48k_len = np.array([len(t) for t in en_tok_48k])

for s, f in [('AVG', np.mean), ('MIN', np.min), ('MAX', np.max), ('q0.999', lambda x: np.quantile(x, q=0.999))]:
    print(f''' {s}
             Ru:                  En:
Symbols:        {f(ru_symbol_len):.4f}     \t     {f(en_symbol_len):.4f}
Tok_8k:         {f(ru_tok_8k_len):.4f}     \t     {f(en_tok_8k_len):.4f}
Tok_16k:        {f(ru_tok_16k_len):.4f}    \t     {f(en_tok_16k_len):.4f}
Tok_32k:        {f(ru_tok_32k_len):.4f}    \t     {f(en_tok_32k_len):.4f}
Tok_48k:        {f(ru_tok_48k_len):.4f}    \t     {f(en_tok_48k_len):.4f}

''')

 AVG
             Ru:                  En:
Symbols:        30.7999     	     31.3712
Tok_8k:         8.8042     	     9.0857
Tok_16k:        7.9527    	     8.6329
Tok_32k:        7.4306    	     8.3837
Tok_48k:        7.2365    	     8.3002


 MIN
             Ru:                  En:
Symbols:        2.0000     	     3.0000
Tok_8k:         1.0000     	     2.0000
Tok_16k:        1.0000    	     2.0000
Tok_32k:        1.0000    	     2.0000
Tok_48k:        1.0000    	     2.0000


 MAX
             Ru:                  En:
Symbols:        841.0000     	     1127.0000
Tok_8k:         304.0000     	     345.0000
Tok_16k:        273.0000    	     319.0000
Tok_32k:        246.0000    	     285.0000
Tok_48k:        231.0000    	     263.0000


 q0.999
             Ru:                  En:
Symbols:        169.0000     	     170.0000
Tok_8k:         58.0000     	     55.0000
Tok_16k:        50.0000    	     49.0000
Tok_32k:        44.0000    	     44.0000
Tok_48k:        41.0000    	     42.0

In [5]:
def get_X_y_by_quantile(tokenizer, X, y, q_X, q_y, add_bos_eos_token=False):
    X_t = [tokenizer.encode(t, out_type=int) for t in X]
    y_t = [tokenizer.encode(t, out_type=int) for t in y]

    X_t_lens = np.array([len(t) for t in X_t])
    y_t_lens = np.array([len(t) for t in y_t])

    X_q = np.quantile(X_t_lens, q=q_X)
    y_q = np.quantile(y_t_lens, q=q_y)

    X_b = X_t_lens <= X_q
    y_b = y_t_lens <= y_q
    mask = X_b & y_b

    X_out = [e for i, e in enumerate(X_t) if mask[i]]

    if add_bos_eos_token:
        y_out = [[tokenizer.bos_id(), *e, tokenizer.eos_id()] for i, e in enumerate(y_t) if mask[i]]
    else:
        y_out = [e for i, e in enumerate(y_t) if mask[i]]

    return X_out, y_out

In [37]:
X_data, y_data = get_X_y_by_quantile(tokenizer16k, dataset[1], dataset[3], 0.999, 0.999, add_bos_eos_token=True)

In [42]:
X_data

[[3552,
  485,
  276,
  1104,
  289,
  2118,
  2894,
  1910,
  260,
  260,
  260,
  1144,
  2261,
  7339,
  260],
 [917, 275, 290, 451, 7295, 305],
 [366, 2062, 792, 1069, 260],
 [416, 298, 2003, 263],
 [416, 298, 2003, 263],
 [416, 298, 2003, 263],
 [416, 282, 263],
 [416, 282, 263],
 [416, 282, 263],
 [416, 282, 263],
 [977,
  3610,
  304,
  517,
  1213,
  261,
  304,
  282,
  573,
  1484,
  1917,
  517,
  1097,
  10395,
  305],
 [502, 3453, 1484, 261, 1917, 517, 1097, 10395, 305],
 [1917, 517, 1097, 10395, 576, 1948, 260],
 [7587, 3885, 614, 4511, 1055, 913, 1166, 1487, 940],
 [314, 306, 462, 2328, 260],
 [314, 306, 462, 2328, 260],
 [314, 306, 462, 2328, 260],
 [314, 306, 462, 2328, 260],
 [341, 398, 261, 275, 304, 575, 260],
 [327, 359, 10124, 6588, 260],
 [327, 359, 10124, 6588, 260],
 [327, 3037, 4954, 263],
 [269, 267, 398, 261, 441, 537, 332, 306, 574, 260],
 [1917,
  2239,
  1214,
  6362,
  329,
  1173,
  290,
  381,
  3690,
  317,
  475,
  442,
  267,
  2381,
  260],
 [2960,

In [38]:
tokenizer16k.decode(X_data[2]), tokenizer16k.decode(y_data[2])

('Мне пора идти спать.', 'I have to go to sleep.')

In [39]:
max(map(len, X_data)), max(map(len, y_data))

(50, 51)

# Dataset

Since the dataset is rather large, you can omit the validation dataset and just use a set of test sentences after the training.

Create a dataset that returns the following
* A pair of tensors `((None, L), (None, P))` -- input sequence of tokens and output sequence of tokens to be fed into decoder (this should start with the BOS token)
* A tensor `(None, P)` -- output sequence of tokens to be predicted (this should end with EOS token)

In [6]:
class DefaultDataset(keras.utils.PyDataset):
    def __init__(self, X, y, tokenizer, shuffle=False, batch_size=128, q_X=0.999, q_y=0.999, seed=1, **kwargs):
        super().__init__(**kwargs)
        self.tokenizer = tokenizer
        self.X = X
        self.y = y
        self.shuffle = shuffle
        self.batch_size = batch_size
        self.rng = np.random.default_rng(seed)

        self.X, self.y = get_X_y_by_quantile(tokenizer, self.X, self.y, q_X, q_y, add_bos_eos_token=True)
        self.idx = np.arange(len(self.X), dtype='int32')
        self.X_lens = np.array([len(t) for t in self.X], dtype=int)
        self.y_lens = np.array([len(t) for t in self.y], dtype=int)

        self.on_epoch_end()

    def __len__(self):
        # return -(-len(self.X) // self.batch_size)
        return len(self.X) // self.batch_size

    def __getitem__(self, ind):
        index = self.idx[ind * self.batch_size: (ind + 1) * self.batch_size]

        ru_lens = self.X_lens[index]
        en_lens = self.y_lens[index]

        max_len_ru = np.max(ru_lens)
        max_len_en = np.max(en_lens)

        seq_ru = keras.utils.pad_sequences(
            [self.X[i] for i in index], 
            maxlen=max_len_ru,
            padding='post',
            truncating='post',
            value=self.tokenizer.pad_id()
        )

        seq_en_no_last = keras.utils.pad_sequences(
            [self.y[i][:-1] for i in index], 
            maxlen=max_len_en - 1,
            padding='post',
            truncating='post',
            value=self.tokenizer.pad_id()
        )

        seq_en = keras.utils.pad_sequences(
            [self.y[i] for i in index], 
            maxlen=max_len_en,
            padding='post',
            truncating='post',
            value=self.tokenizer.pad_id()
        )

        conc_seq = keras.ops.concatenate([seq_ru, seq_en], axis=1)

        loss_mask_ru_bos = keras.ops.zeros((index.shape[0], max_len_ru + 1))

        loss_mask_en_no_bos = keras.ops.broadcast_to(keras.ops.arange(1, max_len_en), (index.shape[0], max_len_en - 1))
        loss_mask_en_no_bos = keras.ops.less(loss_mask_en_no_bos, en_lens[:, None])

        loss_mask = keras.ops.cast(keras.ops.concatenate([loss_mask_ru_bos, loss_mask_en_no_bos], axis=1), 'float32')

        return {'RU_Input': seq_ru, 'EN_Input': seq_en_no_last}, conc_seq[:, 1:], loss_mask[:, 1:]

    def on_epoch_end(self):
        if self.shuffle:
            self.rng.shuffle(self.idx)


In [211]:
train_dataset = DefaultDataset(dataset[1][:10], dataset[3][:10], tokenizer16k, batch_size=3, shuffle=True)

In [212]:
a, b, c = train_dataset[0]

In [213]:
a, b, c

({'ru': array([[ 416,  282,  263,    0,    0,    0],
         [ 917,  275,  290,  451, 7295,  305],
         [ 366, 2062,  792, 1069,  260,    0]], dtype=int32),
  'en': array([[  2, 345, 262, 272, 310, 263,   0,   0],
         [  2, 488, 262, 272, 908, 522, 260,   0],
         [  2, 264, 294, 265, 344, 265, 922, 260]], dtype=int32)},
 <tf.Tensor: shape=(3, 14), dtype=int32, numpy=
 array([[ 282,  263,    0,    0,    0,    2,  345,  262,  272,  310,  263,
            3,    0,    0],
        [ 275,  290,  451, 7295,  305,    2,  488,  262,  272,  908,  522,
          260,    3,    0],
        [2062,  792, 1069,  260,    0,    2,  264,  294,  265,  344,  265,
          922,  260,    3]], dtype=int32)>,
 <tf.Tensor: shape=(3, 14), dtype=float32, numpy=
 array([[0., 0., 0., 0., 0., 0., 1., 1., 1., 1., 1., 1., 0., 0.],
        [0., 0., 0., 0., 0., 0., 1., 1., 1., 1., 1., 1., 1., 0.],
        [0., 0., 0., 0., 0., 0., 1., 1., 1., 1., 1., 1., 1., 1.]],
       dtype=float32)>)

In [ ]:
print(tokenizer16k.decode(a['RU_Input'][1]))
tokenizer16k.decode(a['en'][1])

Давайте что-нибудь попробуем!


"Let's try something."

# Model

Create a model for training. The model should have two inputs: input sequence `(None, L)` and output sequence`(None, P)`. The model output is a single tensor `(None, P)` logits (or probabilities) of the next token predicted for each input one. The model MUST have decoder only architecture.

In [7]:
@keras.saving.register_keras_serializable()
class SwiGLU(keras.layers.Layer):
    def __init__(self, embed_size: int, ff_dim: int, **kwargs):
        super().__init__(**kwargs)

        self.gate_proj = keras.layers.Dense(ff_dim, use_bias=False)
        self.up_proj = keras.layers.Dense(ff_dim, use_bias=False)
        self.down_proj = keras.layers.Dense(embed_size, use_bias=False)

    def call(self, x):
        gate = keras.activations.swish(self.gate_proj(x))
        up = self.up_proj(x)
        return self.down_proj(gate * up)

In [8]:
@keras.saving.register_keras_serializable()
class TransformerBlock(keras.layers.Layer):
    def __init__(self, embed_size, num_heads, ff_dim, dropout, **kwargs):
        super().__init__(**kwargs)
        self.embed_size = embed_size
        self.num_heads = num_heads
        self.ff_dim = ff_dim
        self.dropout = dropout

        self.layer_norm1 = keras.layers.LayerNormalization()
        self.Mha = keras.layers.MultiHeadAttention(num_heads, embed_size // num_heads, dropout=dropout)
        self.dropout1 = keras.layers.Dropout(dropout)

        self.layer_norm2 = keras.layers.LayerNormalization()
        self.swiglu = SwiGLU(embed_size, ff_dim)
        self.dropout2 = keras.layers.Dropout(dropout)

    def call(self, x, attention_mask=None, training=False):
        x_mha = self.layer_norm1(x)
        x_mha = self.Mha(x_mha, x_mha, attention_mask=attention_mask, training=training)

        x_mha = self.dropout1(x_mha, training=training)
        x = x + x_mha

        x_ff = self.layer_norm2(x)
        x_ff = self.swiglu(x_ff)
        x_ff = self.dropout2(x_ff, training=training)

        return x + x_ff

    def compute_output_shape(self, input_shape):
        return input_shape


In [9]:
@keras.saving.register_keras_serializable()
class Transformer(keras.Model):
    def __init__(
        self, 
        vocab_size: int, 
        embed_size: int, 
        max_ru_len: int,
        max_en_len: int, 
        num_blocks: int, 
        num_heads: int, 
        ff_dim: int, 
        dropout: float, 
        pad_id: int, 
        name: str, 
        **kwargs):
        super().__init__(name=name, **kwargs)

        self.vocab_size = vocab_size
        self.embed_size = embed_size
        self.max_ru_len = max_ru_len
        self.max_en_len = max_en_len
        self.num_blocks = num_blocks
        self.num_heads = num_heads
        self.ff_dim = ff_dim
        self.dropout = dropout
        self.pad_id = pad_id
        self.name = name

        self.embedding_layer = keras.layers.Embedding(self.vocab_size, self.embed_size, name='Embedding_layer')
        self.embedding_ru_pos = keras.layers.Embedding(self.max_ru_len, self.embed_size, name='RU_Pos_embedding_layer')
        self.embedding_en_pos = keras.layers.Embedding(self.max_en_len, self.embed_size, name='EN_Embedding_layer')

        self.transformer_blocks = [TransformerBlock(embed_size, num_heads, ff_dim, dropout, name=f'MHA_{i}') for i in range(num_blocks)]

        self.layer_norm = keras.layers.LayerNormalization()

        self.next_token_classifyer = keras.layers.Dense(vocab_size)
        

    def call(self, x, training=False):
        ru = x['RU_Input']
        en = x['EN_Input']
        ru_en = keras.ops.concatenate([ru, en], axis=1)

        pad_mask = keras.ops.not_equal(ru_en, self.pad_id)[:, None, :]

        default_causal_mask = keras.ops.tril(keras.ops.ones((keras.ops.shape(ru_en)[1], keras.ops.shape(ru_en)[1])))
        ru_causal_mask = keras.ops.broadcast_to(keras.ops.arange(0, keras.ops.shape(ru_en)[1]), (1, keras.ops.shape(ru_en)[1]))
        ru_causal_mask = keras.ops.cast(ru_causal_mask < keras.ops.shape(ru)[1], 'float32')

        causal_mask = keras.ops.cast(keras.ops.not_equal(default_causal_mask + ru_causal_mask, 0), dtype='float32')

        # print(causal_mask)
        # print(keras.ops.cast(pad_mask, 'float32'))

        attention_mask = causal_mask * pad_mask

        # print(attention_mask)

        ru_emb = self.embedding_layer(ru)
        en_emb = self.embedding_layer(en)

        ru_pos_emb = self.embedding_ru_pos(keras.ops.arange(0, keras.ops.shape(ru_emb)[1]))
        en_pos_emb = self.embedding_en_pos(keras.ops.arange(0, keras.ops.shape(en_emb)[1]))

        ru_emb = ru_emb + ru_pos_emb
        en_emb = en_emb + en_pos_emb

        x = keras.ops.concatenate([ru_emb, en_emb], axis=1)

        for trans_bl in self.transformer_blocks:
            x = trans_bl(x, attention_mask=attention_mask, training=training)

        x = self.layer_norm(x)
        x = self.next_token_classifyer(x)

        return x

    def compute_loss(
        self,
        x=None,
        y=None,
        y_pred=None,
        sample_weight=None,
        training=True
    ):
        losses = keras.losses.sparse_categorical_crossentropy(
            y,
            y_pred,
            from_logits=True
        )

        if sample_weight is not None:
            losses = losses * sample_weight
            loss = keras.ops.sum(losses) / (keras.ops.sum(sample_weight) + 1e-8)
        else:
            loss = keras.ops.mean(losses)

        return loss

    def get_config(self):
        config = super().get_config()
        config.update({
            'vocab_size': self.vocab_size,
            'embed_size': self.embed_size,
            'max_ru_len': self.max_ru_len,
            'max_en_len': self.max_en_len,
            'num_blocks': self.num_blocks,
            'num_heads': self.num_heads,
            'ff_dim': self.ff_dim,
            'dropout': self.dropout,
            'pad_id': self.pad_id,
        })
        return config


In [10]:
max(map(len, X_data)), max(map(len, y_data))

(50, 51)

In [11]:
MAX_RU_LEN = 50
MAX_EN_LEN = 51

In [14]:
model = Transformer(
    tokenizer16k.vocab_size(),
    128,
    MAX_RU_LEN,
    MAX_EN_LEN,
    6,
    8,
    256,
    0.05,
    tokenizer16k.pad_id(),
    name='model'
)

inputs = {
    'RU_Input': keras.layers.Input((None,), name='RU_Input'),
    'EN_Input': keras.layers.Input((None,), name='EN_Input'),
}
model_w = keras.Model(inputs=inputs, outputs=model(inputs, training=True))

In [15]:
model_w.summary(expand_nested=True)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ EN_Input            │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ RU_Input            │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ model (Transformer) │ (None, None,      │  5,114,368 │ EN_Input[0][0],   │
│                     │ 16000)            │            │ RU_Input[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, None, 128) │  2,048,000 │ -                 │
│ Embedding_layer     │                   │            │                   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 128)       │      6,400 │ -                 │
│ RU_Pos_embedding_l… │                   │            │                   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, 128)       │      6,528 │ -                 │
│ EN_Embedding_layer  │                   │            │                   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └ MHA_0          │ (None, None, 128) │    164,864 │ -                 │
│ (TransformerBlock)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └ MHA_1          │ (None, None, 128) │    164,864 │ -                 │
│ (TransformerBlock)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └ MHA_2          │ (None, None, 128) │    164,864 │ -                 │
│ (TransformerBlock)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └ MHA_3          │ (None, None, 128) │    164,864 │ -                 │
│ (TransformerBlock)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └ MHA_4          │ (None, None, 128) │    164,864 │ -                 │
│ (TransformerBlock)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └ MHA_5          │ (None, None, 128) │    164,864 │ -                 │
│ (TransformerBlock)  │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └                │ (None, None, 128) │        256 │ -                 │
│ layer_normalizatio… │                   │            │                   │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│    └ dense_18       │ (None, None,      │  2,064,000 │ -                 │
│ (Dense)             │ 16000)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 5,114,368 (19.51 MB)

 Trainable params: 5,114,368 (19.51 MB)

 Non-trainable params: 0 (0.00 B)

# Training

Train your model using teacher forcing.

In [16]:
X_train, X_val, y_train, y_val = train_test_split(dataset[1], dataset[3], test_size=0.05)

In [17]:
train_dataset = DefaultDataset(X_train, y_train, tokenizer16k, shuffle=True, batch_size=16)
val_dataset = DefaultDataset(X_val, y_val, tokenizer16k, shuffle=False, batch_size=16)

In [18]:
EPOCHS = 10

In [19]:
STEPS_PER_EPOCH = len(train_dataset)
TOTAL_STEPS = STEPS_PER_EPOCH * EPOCHS

WARMUP_RATIO = 0.1
WARMUP_STEPS = int(TOTAL_STEPS * WARMUP_RATIO)

cosine_decay = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=0.0,
    warmup_target=1e-3,
    warmup_steps=WARMUP_STEPS,
    decay_steps=TOTAL_STEPS - WARMUP_STEPS,
    alpha=0.1
)

model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=cosine_decay)
)

In [20]:
SAVE_DIR = 'models'

def make_callbacks(name):
    return [
        keras.callbacks.ModelCheckpoint(
            filepath=os.path.join(SAVE_DIR, f'{name}.keras'),
            monitor='val_loss',
            mode='min',
            save_best_only=True,
            save_weights_only=False,
            verbose=1
        ),

        keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=4,
            mode='min',
            restore_best_weights=True,
            verbose=1
        )
    ]

In [243]:
model.fit(train_dataset, validation_data=val_dataset, epochs=EPOCHS, callbacks=make_callbacks('model'))

Epoch 1/10
   99/49268 ━━━━━━━━━━━━━━━━━━━━ 2:29:18 182ms/step - loss: 9.6900

KeyboardInterrupt: 

# Testing

Make a function for text translation. Translate some text and evaluate model performance.

In [21]:
model_s = keras.models.load_model('models/model_128_50_51_3_4_512_ep10.keras')
model_b = keras.models.load_model('models/model_180_50_51_8_12_720_ep12.keras')

In [22]:
model_s.summary()

Model: "model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Embedding_layer (Embedding)     │ (1, 50, 128)           │     2,048,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ RU_Pos_embedding_layer          │ (50, 128)              │         6,400 │
│ (Embedding)                     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ EN_Embedding_layer (Embedding)  │ (51, 128)              │         6,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MHA_0 (TransformerBlock)        │ (1, 101, 128)          │       263,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MHA_1 (TransformerBlock)        │ (1, 101, 128)          │       263,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MHA_2 (TransformerBlock)        │ (1, 101, 128)          │       263,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer_normalization_19          │ (1, 101, 128)          │           256 │
│ (LayerNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_28 (Dense)                │ (1, 101, 16000)        │     2,064,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,744,065 (56.24 MB)

 Trainable params: 4,914,688 (18.75 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 9,829,377 (37.50 MB)

In [16]:
def translate(
    text: str,
    tokenizer: 'tokenizer',
    model: keras.Model,
    max_en_len: int=51
) -> str:
    '''Predicts `text`translation using the `model`.

    Arguments:
        text: text to be translated
        tokenizer: tokenizer to use
        model: model ot use
        max_len: maximum length of the prediction (in tokens)

    Returns:
        tranlated text'''
    ...

    ru_tok = np.array(tokenizer.encode(text))[None]
    en_tok = [tokenizer.bos_id()]

    loc_tok = -1
    while loc_tok != tokenizer.eos_id() and len(en_tok) < max_en_len:
        loc_tok = model({'RU_Input': ru_tok, 'EN_Input': np.array(en_tok)[None]})[0][-1]
        loc_tok = int(keras.ops.argmax(loc_tok))
        en_tok.append(loc_tok)

    return tokenizer.decode(en_tok, out_type=str)


In [13]:
test_sentences = [
    "ываыв оорлр укцкукцук мсисмимрпо ропр",
    "Абракадабра",

    "Привет, как дела?",
    "Сколько это стоит?",
    "Я стреляю из лука.",
    "Мне нравится изучать иностранные языки.",
    "Этот код потребляет слишком много памяти.",
    "Я живу в Москве уже десять лет.",
    "Она не пришла на работу, потому что заболела.",
    "Если пойдёт дождь, мы останемся дома и посмотрим фильм.",
    "Мне пришлось отменить встречу, потому что я опоздал на поезд.",
    "Без труда не вытащишь и рыбку из пруда.",
    "Я проснулся ни свет ни заря, чтобы успеть на первый поезд.",
    "Финансовый директор предупредил, что бюджет будет сокращён в следующем квартале.",
    "- Ты придёшь? - Вряд ли.",
    "Дом был построен в 1890 году.",
    "Ключ к успеху - терпение.",
    "Не делай из мухи слона.",
    "Илон Маск посетил Москву.",
]

In [32]:
for text in test_sentences:
    translation = translate(text, tokenizer16k, model_s)
    print(f"RU:                 {text}")
    print(f"EN model:           {translation}\n")

RU:                 ываыв оорлр укцкукцук мсисмимрпо ропр
EN model:           The sheep smark at the Uyghurs of the musicton's music.

RU:                 Абракадабра
EN model:           Abracas are elected.

RU:                 Привет, как дела?
EN model:           Hello, how are you?

RU:                 Сколько это стоит?
EN model:           How much does it cost?

RU:                 Я стреляю из лука.
EN model:           I'm shooting at the bow.

RU:                 Мне нравится изучать иностранные языки.
EN model:           I like studying foreign languages.

RU:                 Этот код потребляет слишком много памяти.
EN model:           This device consumes too much memory.

RU:                 Я живу в Москве уже десять лет.
EN model:           I've been living in Moscow for ten years.

RU:                 Она не пришла на работу, потому что заболела.
EN model:           She didn't come to work because she got sick.

RU:                 Если пойдёт дождь, мы останемся дома и 

In [33]:
for text in test_sentences:
    translation = translate(text, tokenizer16k, model_b)
    print(f"RU:                 {text}")
    print(f"EN model:           {translation}\n")

RU:                 ываыв оорлр укцкукцук мсисмимрпо ропр
EN model:           The gums are covered by a dwarf of the intersection of the drug slop.

RU:                 Абракадабра
EN model:           Abracada is notified.

RU:                 Привет, как дела?
EN model:           Hi, how are you?

RU:                 Сколько это стоит?
EN model:           How much does it cost?

RU:                 Я стреляю из лука.
EN model:           I'm shooting the onions.

RU:                 Мне нравится изучать иностранные языки.
EN model:           I like studying foreign languages.

RU:                 Этот код потребляет слишком много памяти.
EN model:           This code is consumed too much memory.

RU:                 Я живу в Москве уже десять лет.
EN model:           I've lived in Moscow for ten years.

RU:                 Она не пришла на работу, потому что заболела.
EN model:           She didn't come to work because she fell ill.

RU:                 Если пойдёт дождь, мы останемся 

# Bonus

Train the model using its own predictions as inputs. The probability to use the predicted tokens instead of the ones from the dataset should increase as the training progresses. Train for one or two epochs. Evaluate the model, does the model perform better?

In [36]:
model_s_train = keras.models.load_model('models/model_128_50_51_3_4_512_ep10.keras')
# model_b_train = keras.models.load_model('models/model_180_50_51_8_12_720_ep12.keras')

In [37]:
train_dataset = DefaultDataset(dataset[1], dataset[3], tokenizer16k, shuffle=True, batch_size=16)

In [38]:
len(train_dataset[0])

3

In [39]:
EPOCHS = 2

STEPS_PER_EPOCH = len(train_dataset)
TOTAL_STEPS = STEPS_PER_EPOCH * EPOCHS

WARMUP_RATIO = 0.1
WARMUP_STEPS = int(TOTAL_STEPS * WARMUP_RATIO)

cosine_decay = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=0.0,
    warmup_target=1e-3,
    warmup_steps=WARMUP_STEPS,
    decay_steps=TOTAL_STEPS - WARMUP_STEPS,
    alpha=0.1
)

optimizer = keras.optimizers.AdamW(learning_rate=cosine_decay)
loss = keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')

In [ ]:
def train_step_wrapper(model, optimizer, loss, eos_token_id, pad_token_id):
    @tf.function(reduce_retracing=True)
    def train_step(ds, prob):
        ru_X = ds[0]['RU_Input']
        en_X = ds[0]['EN_Input']
        y = ds[1]
        loss_mask = ds[2]

        batch_size = keras.ops.shape(ru_X)[0]

        en_new = en_X[:, :1]

        for i in tf.range(1, tf.shape(en_X)[1]):
            en_pred = keras.ops.argmax(model({'RU_Input': ru_X, 'EN_Input': en_new}, training=False)[:, -1], axis=-1)[:, None]
            en_true = en_X[:, i][:, None]

            r = keras.random.uniform((batch_size, 1))
            en_add2new = keras.ops.where((r > prob) | (en_true == eos_token_id) | (en_true == pad_token_id), en_true, en_pred)

            en_new = keras.ops.concatenate([en_new, en_add2new], axis=1)

        with tf.GradientTape() as tape:
            pred = model({'RU_Input': ru_X, 'EN_Input': en_new}, training=True)
            loss_val = tf.reduce_sum(loss(y, pred) * loss_mask) / tf.reduce_sum(loss_mask * 1e-8)
            
        gradients = tape.gradient(
            loss_val,
            model.trainable_variables
        )

        optimizer.apply_gradients(
            zip(gradients, model.trainable_variables)
        )

        return loss_val
        
    return train_step

In [43]:
MAX_PROP = 0.7
tr = train_step_wrapper(model_s_train, optimizer, loss, tokenizer16k.eos_id(), tokenizer16k.pad_id())

steps = 0
for epoch in range(EPOCHS):
    for b in tqdm(train_dataset, desc=f'Эпоха {epoch}'):
        tr(b, MAX_PROP * steps / (TOTAL_STEPS - 1))

        steps += 1

    print(model.evaluate(val_dataset))
    model.save(f'models/model_128_50_51_3_4_512_ep10_finetuned_ep{epoch + 1}.keras')

Эпоха 0:   0%|          | 0/51860 [00:00<?, ?it/s]W0000 00:00:1795676667.000248    7501 local_rendezvous.cc:412] Local rendezvous is aborting with status: INVALID_ARGUMENT: seed must have shape [2], not [0]
	 [[{{function_node while_body_975613}}{{node while/stateless_random_uniform/StatelessRandomGetKeyCounter}}]]
Эпоха 0:   0%|          | 0/51860 [00:08<?, ?it/s]


InvalidArgumentError: Graph execution error:

Detected at node while/stateless_random_uniform/StatelessRandomGetKeyCounter defined at (most recent call last):
  File "/home/maks/.local/share/uv/python/cpython-3.13.10-linux-x86_64-gnu/lib/python3.13/runpy.py", line 198, in _run_module_as_main

  File "/home/maks/.local/share/uv/python/cpython-3.13.10-linux-x86_64-gnu/lib/python3.13/runpy.py", line 88, in _run_code

  File "/home/maks/HW/NLP/.venv/lib/python3.13/site-packages/ipykernel_launcher.py", line 18, in <module>

  File "/home/maks/HW/NLP/.venv/lib/python3.13/site-packages/traitlets/config/application.py", line 1080, in launch_instance

  File "/home/maks/HW/NLP/.venv/lib/python3.13/site-packages/ipykernel/kernelapp.py", line 768, in start

  File "/home/maks/HW/NLP/.venv/lib/python3.13/site-packages/tornado/platform/asyncio.py", line 203, in start

  File "/home/maks/.local/share/uv/python/cpython-3.13.10-linux-x86_64-gnu/lib/python3.13/asyncio/base_events.py", line 683, in run_

In [14]:
model_s = keras.models.load_model('models/model_128_50_51_3_4_512_ep10.keras')
model_s_ep1 = keras.models.load_model('models/model_128_50_51_3_4_512_ep10_finetuned_ep1.keras')
model_s_ep2 = keras.models.load_model('models/model_128_50_51_3_4_512_ep10_finetuned_ep2.keras')

model_b = keras.models.load_model('models/model_180_50_51_8_12_720_ep12.keras')
model_b_ep1 = keras.models.load_model('models/model_180_50_51_8_12_720_ep12_finetuned_ep1.keras')

In [17]:
for text in test_sentences:
    print(f"RU:                    {text}")
    print(f"EN model_s 10ep:       {translate(text, tokenizer16k, model_s)}")
    print(f"EN model_s 10ep 1ep:   {translate(text, tokenizer16k, model_s_ep1)}")
    print(f"EN model_s 10ep 2ep:   {translate(text, tokenizer16k, model_s_ep2)}\n")
    print(f"EN model_b 12ep:       {translate(text, tokenizer16k, model_b)}")
    print(f"EN model_b 12ep 1ep:   {translate(text, tokenizer16k, model_b_ep1)}")
    print('---------------------------------------------------------------------------')

RU:                    ываыв оорлр укцкукцук мсисмимрпо ропр
EN model_s 10ep:       The sheep smark at the Uyghurs of the musicton's music.
EN model_s 10ep 1ep:   The surroundsssssssssurtssurturturs.
EN model_s 10ep 2ep:   Thessssssssssssssss a.

EN model_b 12ep:       The gums are covered by a dwarf of the intersection of the drug slop.
EN model_b 12ep 1ep:   The snor of the rotsssss the the arp.
---------------------------------------------------------------------------
RU:                    Абракадабра
EN model_s 10ep:       Abracas are elected.
EN model_s 10ep 1ep:   Abracasbra are elected.
EN model_s 10ep 2ep:   Abra is elected.

EN model_b 12ep:       Abracada is notified.
EN model_b 12ep 1ep:   Abracadabra is abra.
---------------------------------------------------------------------------
RU:                    Привет, как дела?
EN model_s 10ep:       Hello, how are you?
EN model_s 10ep 1ep:   Hello, how are you?
EN model_s 10ep 2ep:   Hello, how are you?

EN model_b 12ep:    